# 1. Plan

## Environment setup

- Test jupyter kernel is working.
- Ensure libraries are imported.

### Test Kernel is working

In [ ]:
print ("Hello, World!")

# 2. Ingestion of Weather Data

- Get data via url.
- Paramatise the url to get based on wether type and region.
- Parse into DataFrame.

### Import relevant libraries

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## 2.1 Plan

## 2. Get data from url

### 3. Attempt to call url to download data

In [ ]:
url= "https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/Sunshine/date/East_Anglia.txt"
response = requests.get(url)
print (response.text)

### 3. Parameterise the url

In [ ]:
# This gives a url like this
parameter = "Sunshine"
region = "East_Anglia"
url= f"https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{parameter}/date/{region}.txt"
response = requests.get(url)
print(response.text)

## 2. Files based ingestion

### 3. Check file exists

In [ ]:
# First, let's import os to check if file exists
import os

raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module1_project/data/weather/sunshine/east_anglia_sun.txt'
print(f"File exists: {os.path.exists(raw_file_path)}")

# If file exists, let's try to read first few lines directly
if os.path.exists(raw_file_path):
    with open(raw_file_path, 'r') as file:
        print("\nFirst few lines of the file:")
        for i, line in enumerate(file):
            if i < 3:  # Print first 3 lines
                print(line.strip())

### 3. Present first five rows of file to assess for processing

In [ ]:
# Read the raw text file
with open(raw_file_path, 'r') as file:
    print("First 5 lines of raw file:")
    for i, line in enumerate(file):
        if i < 5:
            print(f"Line {i+1}: {repr(line)}")  # repr() shows whitespace characters

This indicates the first five rows of data are only **Contextual information** and not part of the dataset proper. This is not needed and will need to be removed for the purposes of data analysis.

In [ ]:
# Explore more rows to identify where the data starts
with open(raw_file_path, 'r') as file:
    print("First 10 lines of raw file:")
    for i, line in enumerate(file):
        if i < 10:
            print(f"Line {i+1}: {repr(line)}")  # repr() shows whitespace characters

The first row of data starts at Line 7, with the headers of columns on Line 6.

## First five lines of actual data

### Headers

In [ ]:
# Read the raw text file
with open(raw_file_path, 'r') as file:
    print("Headers of columns:")
    for i, line in enumerate(file):
        if i == 5:
            print(f"Line {i+1}: {repr(line)}")  # repr() shows whitespace characters

### First five lines of actual data

In [ ]:
# Read the raw text file
with open(raw_file_path, 'r') as file:
    print("First 5 lines of raw file:")
    for i, line in enumerate(file):
        if i > 5 and i < 10:
            print(f"Line {i+1}: {repr(line)}")  # repr() shows whitespace characters

## Try reading in

Direct reading of file as csv with pandas

In [ ]:
# Try reading with explicit encoding and error handling
try:
    df = pd.read_csv(raw_file_path, encoding='utf-8')
    print("\nDataframe info:")
    print(df.info())
    print("\nFirst 5 rows:")
    print(df.head())
except Exception as e:
    print(f"Error: {str(e)}")

This shows that the data cannot be read in. Suggesting that pandas is unable to determine the correct delimiter in the CSV file. This is down to the file actually being a .txt file, and being delimited by both tabs and spaces.

Attempt to handle variable spacing and whitespace

In [ ]:
try:
    # Read with flexible whitespace handling
    df = pd.read_csv(raw_file_path, 
                     sep='\s+',  # Handle variable spacing
                     skipinitialspace=True,   # Skip leading spaces
                     skip_blank_lines=True)   # Skip empty lines
    
    print("\nDataframe info:")
    print(df.info())
    print("\nFirst 5 rows:")
    print(df.head())
except Exception as e:
    print(f"Error: {str(e)}")

This appears to fail as it is expecting fewer columns than are presented in the data table proper. This is likely due to the contextual inforamtion presented at the start of the file.

Try with text file where contextual data has been stripped out

Check file exists

In [ ]:
# First, let's import os to check if file exists
import os

stripped_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module1_project/data/weather/sunshine/east_anglia_sun_stripped.txt'
print(f"File exists: {os.path.exists(stripped_file_path)}")

# If file exists, let's try to read first few lines directly
if os.path.exists(stripped_file_path):
    with open(stripped_file_path, 'r') as file:
        print("\nFirst few lines of the file:")
        for i, line in enumerate(file):
            if i < 3:  # Print first 3 lines
                print(line.strip())

This shows the first three line of the file. The first being the column headers, and then two rows of data.

Try just reading straight into pandas

In [ ]:
# Try reading with explicit encoding and error handling
try:
    df = pd.read_csv(stripped_file_path, encoding='utf-8')
    print("\nDataframe info:")
    print(df.info())
    print("\nFirst 5 rows:")
    print(df.head())
except Exception as e:
    print(f"Error: {str(e)}")

See how this is handled with the additional parameters stripping out the whitespace

In [ ]:
try:
    # Read with flexible whitespace handling
    df = pd.read_csv(stripped_file_path, 
                     sep='\s+',  # Handle variable spacing
                     skipinitialspace=True,   # Skip leading spaces
                     skip_blank_lines=True)   # Skip empty lines
    
    print("\nDataframe info:")
    print(df.info())
    print("\nFirst 5 rows:")
    print(df.head())
except Exception as e:
    print(f"Error: {str(e)}")

# 1. Process whole file

## Read raw file into a pandas dataframe

It's clear that the metadata in each of these files is uniform, and always occupies the first five rows. Using the "skiprows" parameter allows loading of the file whilst skipping the five metadata rows.

In [ ]:
# Path to data: /home/lnx_workspaces/bpp_projects/bpp_module1_project/data/weather/sunshine/east_anglia_sun.csv
df = pd.read_csv(raw_file_path, sep="\s+", skiprows=5, header=0)
df.head(5)

**N.B.** have replaced the paramater definition 'delim_whitespace' keyword with "sep='\s+'" as the former is depreciated, as indicated in execution Futurewarning hints.

# 1. Process all files using Dynamic URL

Rather than downloading each of file for each region for each parameter, it would be better to directly retrieve the data from the API or URL.

The URL clearly indicates in plain text the Region and Parameter lables. I can extract these and inject them into the URL string used to request the data.

## Parmeter Lables

|Parameter|Label|
|---------|-----|
|Max temp|Tmax|
|Min temp|Tmin|
|Mean temp|Tmean|
|Sunshine|Sunshine|
|Rainfall|Rainfall|
|Rain days ≥1.0mm|Raindays1mm|
|Days of air frost|AirFrost|

In [ ]:
data_weather = ['Tmax','Tmin','Tmean','Sunshine','Rainfall','Raindays1mm','AirFrost']

## Regions Labels

I can extract each of the Region names from the download page source. This gives me the following table.

### Each of the Regions

|Region|Label|
|------|-----|
|UK|UK|
|England|England|
|Wales|Wales|
|Scotland|Scotland|
|Northern Ireland|Northern_Ireland|
|England & Wales|England_and_Wales|
|England N|England_N|
|Englan S|Englan_S|
|Scotland N|Scotland_N|
|Scotland E|Scotland_E|
|Scotland W|Scotland_W|
|England E & NE|England_E_and_NE|
|England NW/Wales N|England_NW_and_N_Wales|
|Midlands|Midlands|
|East Anglia|East_Anglia|
|England SW/Wales S|England_SW_and_S_Wales|
|England SE/Central S|England_SE_and_Central_S|

In [ ]:
data_region = ['UK','England','Wales','Scotland','Northern_Ireland','England_and_Wales','England_N','Englan_S','Scotland_N','Scotland_E','Scotland_W','England_E_and_NE','England_NW_and_N_Wales','Midlands','East_Anglia','England_SW_and_S_Wales','England_SE_and_Central_S']